# Moving average and ARMA models: lab notebook

**Statistical Modelling and Forecasting, ARIMA and vector autoregression**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.arima_process import ArmaProcess
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox

rng = np.random.default_rng(5)
draw = rng.standard_normal

### 1. MA(1): theoretical ACF cuts off after lag 1

In [ ]:
# 1. MA(1): theoretical ACF cuts off after lag 1
theta = 0.8
ma = ArmaProcess([1], [1, theta])
print("MA(1) theory rho_1:", round(ma.acf(4)[1], 4), "= theta/(1+theta^2) =", round(theta / (1 + theta ** 2), 4), "| rho_2, rho_3:", np.round(ma.acf(4)[2:], 4))
print("invertible:", ma.isinvertible, "| for theta = 1.25:", ArmaProcess([1], [1, 1.25]).isinvertible)
print("rho_1 of theta=1.25 and of 1/1.25 = 0.8:", round(ArmaProcess([1], [1, 1.25]).acf(2)[1], 4), round(ArmaProcess([1], [1, 0.8]).acf(2)[1], 4))

### 2. ARMA(1,1): phi = 0.6, theta = 0.5, both ACF and PACF tail off

In [ ]:
# 2. ARMA(1,1): phi = 0.6, theta = 0.5, both ACF and PACF tail off
arma = ArmaProcess([1, -0.6], [1, 0.5])
print("ARMA(1,1) theory ACF 1..5:", np.round(arma.acf(6)[1:], 3))
print("ARMA(1,1) theory PACF 1..5:", np.round(arma.pacf(6)[1:], 3))
print("psi weights:", np.round(arma.arma2ma(6), 3))

### 3. Fit it back from a simulated sample

In [ ]:
# 3. Fit it back from a simulated sample
y = arma.generate_sample(nsample=400, distrvs=draw)
res = ARIMA(y, order=(1, 0, 1), trend="c").fit()
print(res.summary().tables[1])

### 4. Residual check and forecasts (forecast error variance follows the psi weights)

In [ ]:
# 4. Residual check and forecasts (forecast error variance follows the psi weights)
print("Ljung-Box p-value (10 lags):", round(float(acorr_ljungbox(res.resid, lags=[10], model_df=2)["lb_pvalue"].iloc[0]), 3))
fc = res.get_forecast(5)
print("5-step forecast:", np.round(fc.predicted_mean, 3))
print("std errors:", np.round(fc.se_mean, 3))

### 5. The numerical problem: MA(1) with theta = 0.5, sigma^2 = 4, mu = 10, e_T = 2

In [ ]:
# 5. The numerical problem: MA(1) with theta = 0.5, sigma^2 = 4, mu = 10, e_T = 2
th, s2, mu, eT = 0.5, 4, 10, 2
print("variance:", s2 * (1 + th ** 2), "| rho_1:", round(th / (1 + th ** 2), 4), "| 1-step forecast:", mu + th * eT, "| 2-step forecast:", mu)
ph, t2 = 0.6, 0.4
r1 = (1 + ph * t2) * (ph + t2) / (1 + 2 * ph * t2 + t2 ** 2)
print("ARMA(1,1) rho_1:", round(r1, 4), "rho_2:", round(ph * r1, 4))

## Expected output

Running all cells above should print:

```
MA(1) theory rho_1: 0.4878 = theta/(1+theta^2) = 0.4878 | rho_2, rho_3: [0. 0.]
invertible: True | for theta = 1.25: False
rho_1 of theta=1.25 and of 1/1.25 = 0.8: 0.4878 0.4878
ARMA(1,1) theory ACF 1..5: [0.773 0.464 0.278 0.167 0.1  ]
ARMA(1,1) theory PACF 1..5: [ 0.773 -0.332  0.16  -0.08   0.04 ]
psi weights: [1.    1.1   0.66  0.396 0.238 0.143]
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.1945      0.192     -1.014      0.311      -0.570       0.181
ar.L1          0.5983      0.046     13.090      0.000       0.509       0.688
ma.L1          0.5924      0.047     12.538      0.000       0.500       0.685
sigma2         0.9176      0.066     13.936      0.000       0.789       1.047
==============================================================================
Ljung-Box p-value (10 lags): 0.065
5-step forecast: [-1.403 -0.917 -0.627 -0.453 -0.349]
std errors: [0.958 1.49  1.638 1.689 1.706]
variance: 5.0 | rho_1: 0.4 | 1-step forecast: 11.0 | 2-step forecast: 10
ARMA(1,1) rho_1: 0.7561 rho_2: 0.4537
```

## Practice exercises

Try these in the cells below.

1. Set the MA part to θ = 0.8 and AR to zero. The ACF should stop after lag 1. What is ρ₁ in theory and in the sample?
2. Keep φ = 0.7 and add θ = −0.7. The ARMA(1,1) nearly cancels. What does the series look like?
3. Choose θ = 1.2 and read the invertibility warning. Then use 1/1.2 and compare the theoretical ACF values.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
